## Tool definition

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\tools\base.py:36: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [13]:
@tool("square",description="Calculate the square of a number")
def tool2(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [5]:
tool1.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [8]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [10]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately **21.61018278497431**.


In [11]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='c6e2d6d2-7541-480f-aca0-ec348e1e1232'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to compute sqrt(467). Use the function.', 'tool_calls': [{'id': 'fc_16aa7893-2c4b-40e5-9573-52a264ad614c', 'function': {'arguments': '{"x":467}', 'name': 'square_root'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 153, 'total_tokens': 189, 'completion_time': 0.051636823, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.007556767, 'prompt_tokens_details': None, 'queue_time': 0.211784519, 'total_time': 0.05919359}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a129c6-46bd-76e2-ade6-f3a6fec7bf43-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}

In [12]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'fc_16aa7893-2c4b-40e5-9573-52a264ad614c', 'type': 'tool_call'}]


In [14]:
agent=create_agent(
    model=model,
    tools=[tool1, tool2],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [21]:
question = HumanMessage(content="What is the square root of 467 and the square of 23?")
response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

- The square root of **467** is approximately **21.61018278497431**.  
- The square of **23** is **529**.


In [30]:
for msg in response["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls is None and hasattr(msg, "additional_kwargs"):
        tool_calls = msg.additional_kwargs.get("tool_calls")
    if tool_calls:
        print(tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'fc_6233931a-a0aa-457b-8a20-3b4e154cc323', 'type': 'tool_call'}]
[{'name': 'square', 'args': {'x': 23}, 'id': 'fc_e1e7a18a-4840-4828-a890-cd05080410f4', 'type': 'tool_call'}]
